## Run DeepEval LLM-as-judge with local LLMs of Ollama

In [ ]:

from deepeval.test_case import LLMTestCase;
from deepeval import evaluate;
from deepeval.metrics import ExactMatchMetric

test_case = LLMTestCase(
    input="What is the capital of France?",
    expected_output="The capital of France is Paris.",
    actual_output="The capital of France is Paris."
)

evaluate([test_case], metrics = [
    ExactMatchMetric()
])

## Test Code LLM as-judge - AnswerRelevancy

In [ ]:
!deepeval set-ollama --model llama3.2:latest

In [ ]:
from dotenv import load_dotenv

from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate.configs import AsyncConfig
from deepeval.models import OllamaModel
import os


load_dotenv()

ollama_model = OllamaModel( model=os.getenv("LOCAL_OLLAMA_MODEL"), base_url=os.getenv("LOCAL_OLLAMA_BASE_URL"))
    


def test_answer_relevancy():
    test_case = LLMTestCase(
        input="What is the capital of France?",
        expected_output="The capital of France is Paris.",
        actual_output="Paris.",
    )

    metric = AnswerRelevancyMetric(
        model=ollama_model,   # use Ollama instead of the default OpenAI
        threshold=0.5,
    )

    evaluate(
        test_cases=[test_case],
        metrics=[metric],
        async_config=AsyncConfig(run_async=False),
    )


if __name__ == "__main__":
    test_answer_relevancy()

## Test Code with thinking model in place with OllamaModel Class of DeepEval

In [ ]:
from deepeval.test_case import LLMTestCase;
from deepeval import evaluate;
from deepeval.metrics import AnswerRelevancyMetric
from dotenv import load_dotenv, find_dotenv
from deepeval.evaluate import AsyncConfig
from deepeval.models import OllamaModel
import os

load_dotenv(find_dotenv())

ollama_model = OllamaModel(
    model="llama3.2:latest",
    base_url="http://localhost:11434"
)

test_case = LLMTestCase(
    input="What is the capital of France?",
    expected_output="The capital of France is Paris.",
    actual_output="Chennai."
)

evaluate(test_cases =[test_case], 
         metrics = [AnswerRelevancyMetric(model=ollama_model)],
         async_config=AsyncConfig(run_async=False)
)

## Test Code by Disabling the thinking of model in place with OllamaModel Class of DeepEval

In [ ]:
from deepeval.test_case import LLMTestCase;
from deepeval import evaluate;
from deepeval.metrics import AnswerRelevancyMetric
from dotenv import load_dotenv, find_dotenv
from deepeval.evaluate import AsyncConfig
from deepeval.models import OllamaModel
import os
from typing import Optional, Tuple, Union
from pydantic import BaseModel

load_dotenv(find_dotenv())


class OllamaModelNoThink(OllamaModel):
     def generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model()
        messages = [{"role": "user", "content": prompt}]

        response = chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

     async def a_generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model(async_mode=True)
        messages = [{"role": "user", "content": prompt}]

        response = await chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

ollama_model = OllamaModelNoThink(
     model="llama3.2:latest",
     base_url="http://localhost:11434"
)

test_case = LLMTestCase(
    input="What is the capital of France?",
    expected_output="The capital of France is Paris.",
    actual_output="Chennai."
)

evaluate(test_cases =[test_case], 
         metrics = [AnswerRelevancyMetric(model=ollama_model)],
         async_config=AsyncConfig(run_async=False)
)